# Composition

**Capabilities are verbs in one registry, and every surface is a projection of it.** The CLI, the HTTP API, the manual, the planner and every client read the same registry, so a capability added once appears in all of them and none of them can drift.

> **Every cell in this notebook runs.** They are generated from
> [`tools/notebooks/spec.py`](../tools/notebooks/spec.py) and executed by CI, so a
> cell that cannot run does not reach a commit. Change a cell, re-run it, and the
> page is yours — that is what it is for.


The model is the shell: small verbs, piped together, producing results no single
verb could. What makes this different from `ls | grep | wc` is what travels
through the pipe.

A shell pipe carries bytes and loses everything else — which is why
`curl | jq | grep` cannot tell you *why* a value is what it is. Here the pipe
carries a **`Flow`**: a value, its kind, its reasoning path, and its gaps.
Composing accumulates the explanation instead of discarding it.

In [ ]:
# --- setup: works locally, on Binder, and on Colab -------------------------
import subprocess, sys, pathlib

def _ensure_installed():
    """Install the package if it is not importable. No-op when it already is."""
    try:
        import slpie, gratimos          # noqa: F401
        return pathlib.Path(slpie.__file__).parent.parent
    except ModuleNotFoundError:
        pass
    here = pathlib.Path.cwd()
    root = next(
        (p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), None,
    )
    if root is None:                     # Colab: no checkout, so fetch one
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Reimain/Macropol-s.git", "/content/Macropol-s"],
            check=True,
        )
        root = pathlib.Path("/content/Macropol-s")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)],
                   check=True)
    sys.path.insert(0, str(root))
    return root

ROOT = _ensure_installed()
print("package root:", ROOT)

import slpie
print("slpie", slpie.__version__)

## The registry

Every verb declares what it consumes and what it produces.

In [ ]:
from slpie.compose import Composition, Context, Kind, registry

verbs = registry()
verb = verbs.require("govern")

print("name:     ", verb.name)
print("group:    ", verb.group)
print("consumes: ", verb.consumes.label)
print("produces: ", verb.produces.label)
print("summary:  ", verb.summary)
print()
print("parameters:")
for param in verb.params:
    mark = " (required)" if param.required else ""
    print(f"  --{param.name:14}{param.type:8}{mark}  {param.help[:44]}")

## Typing the pipe is what makes the rest possible

Because every verb declares its kinds, an impossible composition is refused **before anything runs** — not halfway through, with a stack trace and a partially changed environment.

In [ ]:
bad = Composition.read("findings | attach", verbs=verbs)
check = bad.validate()

print("valid:", check.ok)
print()
print(check.explain())

Both kinds are named. Nothing executed — and that matters, because `attach` changes the environment.

## What can follow what

"What can I pipe into this?" is a query over the registry, not prose somebody maintains.

In [ ]:
print("verbs that can follow `link`:")
print("  ", " ".join(sorted(v.name for v in verbs.successors("link"))[:14]))
print()
print("verbs that can start a pipeline (they consume NOTHING):")
print("  ", " ".join(sorted(v.name for v in verbs.sources())[:14]))
print()
print("kinds actually reachable in <= 4 stages:")
print("  ", " ".join(sorted(verbs.reachable())))

## Build something and watch the provenance accumulate

In [ ]:
import json, pathlib, tempfile

WORK = pathlib.Path(tempfile.mkdtemp(prefix="slpie-nb-"))
(WORK / "package.json").write_text(json.dumps({
    "name": "demo", "version": "1.0.0", "dependencies": {"lodash": "^3.0.0"},
}))
(WORK / "package-lock.json").write_text(json.dumps({
    "name": "demo", "lockfileVersion": 3,
    "packages": {"node_modules/lodash": {"version": "4.17.21"}},
}))

# A manifest asking for ^3 and a lockfile pinning 4.17.21 — they contradict.
stages = ["discover " + str(WORK), "link", "findings"]
for count in range(1, len(stages) + 1):
    pipeline = " | ".join(stages[:count])
    result = Composition.read(pipeline, verbs=verbs).run(Context(root=str(WORK)))
    flow = result.flow
    print(f"{pipeline[-42:]:44} {flow.kind.label:14} "
          f"steps={len(flow.reasoning.steps):2} gaps={len(flow.gaps)}")

The reasoning path grows at every stage and nothing is dropped. A gap raised at stage one is still there at stage three — that is invariant 5 ("every answer carries its reasoning and its gaps") holding *through composition*, which is what makes long pipelines trustworthy rather than merely convenient.

In [ ]:
result = Composition.read(
    f"discover {WORK} | link | findings", verbs=verbs,
).run(Context(root=str(WORK)))

print(result.flow.reasoning.render()[:900])

## Explaining before running

You can see what a composition will do, and what it will cost, while the decision is still free.

In [ ]:
print(Composition.read(f"discover {WORK} | link | constraints | findings",
                       verbs=verbs).explain())

## Shaping verbs work on typed objects, not text

`filter`, `sort`, `head` and `unique` accept `ANY` and produce `SAME`, so they slot in anywhere. They compare a `Severity`, not a rendering of one — which is why `grep` could not do this job.

In [ ]:
result = Composition.read(
    f"discover {WORK} | link | findings | sort --field severity --desc | head --count 3",
    verbs=verbs,
).run(Context(root=str(WORK)))

print("stages:", " → ".join(result.flow.stages))
print("kind:  ", result.flow.kind.label, "(unchanged by the shaping verbs)")
print("count: ", result.flow.size)

## Your turn

Try these — each one is a real, valid composition:

```python
Composition.read(f"discover {WORK} | reason | ask --question 'what is risky?'", verbs=verbs)
Composition.read(f"discover {WORK} | sbom --format cyclonedx", verbs=verbs)
Composition.read("audit | verdicts --only violated", verbs=verbs)
```

And try an invalid one — `validate()` will name both kinds.

In [ ]:
# Scratch cell — edit and run.
pipeline = f"discover {WORK} | reason | ask --question 'what should I fix first?'"

result = Composition.read(pipeline, verbs=verbs).run(Context(root=str(WORK)))
print("ok:", result.ok, "| kind:", result.flow.kind.label)
print(result.flow.facts.get("answer", "")[:600])